# Tokenizzazione subword

## Dalla tokenizzazione tradizionale ai modelli linguistici

Nel notebook 02 abbiamo scelto confini tra parole e punteggiatura, confrontato convenzioni e tokenizer, e ragionato su occorrenze, tipi e parole fuori vocabolario (OOV). Qui non ripetiamo quelle procedure: osserviamo come un modello possa rappresentare una parola rara o mai incontrata usando **pezzi riutilizzabili**.


### Lettura di riferimento

Daniel Jurafsky e James H. Martin, *Speech and Language Processing*, 3a edizione, [Capitolo 2: Words and Tokens](https://web.stanford.edu/~jurafsky/slp3/2.pdf). Per questa lezione: introduzione, sezioni 2.1 **Words**, 2.2 **Morphemes**, introduzione a 2.3 **Unicode**, 2.4 **BPE** e 2.6 **Corpora**.



## 1. Che cosa resta da risolvere?

Nel notebook 02 abbiamo visto che il vocabolario dipende da come segmentiamo e normalizziamo il testo. Con token interi, due forme simili possono diventare tipi distinti e una parola mai osservata può essere OOV.

La tokenizzazione subword non sostituisce la tokenizzazione linguistica per ogni compito: offre ai modelli un vocabolario finito di unità riutilizzabili, combinabili per rappresentare molte parole. I segmenti appresi non coincidono necessariamente con morfemi o sillabe.

### Esercizio 1 — previsione e confronto con OOV

Considerate il piccolo corpus: `gatto gatti gattino`, `casa casetta`, `gatto casa`.

1. Elencate i tipi di parola intera osservati e indicate quali tra `gattini`, `casette`, `cane` sarebbero OOV a livello di parola.
2. Proponete una segmentazione subword plausibile per queste tre forme, senza ancora applicare un algoritmo.
3. Quali pezzi potrebbero essere riutilizzati? Quali segmentazioni sono solo ipotesi e non conoscenza linguistica garantita?
4. Collegate la risposta ai concetti di tipo, occorrenza e OOV del notebook 02.


## 2. Parole, token e unità di analisi

Una **parola** è un'unità linguistica, ma i suoi confini non sono sempre ovvi: apostrofi, composti, clitici e convenzioni ortografiche possono rendere diverse le analisi. Un **token** è invece un'unità prodotta da una specifica procedura di segmentazione; può coincidere con una parola, includere punteggiatura o essere un frammento subword. Nel notebook 02 avete già confrontato convenzioni di tokenizzazione: qui ci interessa cosa succede quando l'unità utile al modello è più piccola della parola.

Distinguete inoltre l'occorrenza di un token nel testo dal tipo distinto rappresentato nel vocabolario. La stessa forma può ricorrere molte volte; la normalizzazione e il tokenizer determinano quali occorrenze vengono ricondotte allo stesso tipo.

### Esercizio 2 — parola, occorrenza, tipo o subword?

Per `Le ragazze hanno riletto i libri.` individuate: le parole ortografiche secondo una convenzione ragionevole, le occorrenze di token, i tipi distinti dopo conversione in minuscolo e una possibile segmentazione subword di `riletto`.

Poi confrontate `l'acqua` e `l’acqua`: riprendete le convenzioni discusse nel notebook 02 e spiegate perché non esiste una risposta unica senza dichiarare il criterio. Non usate `split()` come definizione universale di parola.


## 3. Morfemi e subword

Un **morfema** è una minima unità linguistica dotata di significato o funzione grammaticale. Alcuni morfemi possono comparire da soli (per esempio `casa`), altri sono legati a una base: prefissi e suffissi come `ri-`, `-zione`, `-mente` possono contribuire al significato o alla grammatica. Le parole possono contenere una radice e uno o più affissi; la struttura varia da lingua a lingua.

Le unità **subword** apprese da un tokenizer sono invece il risultato di un algoritmo e di un corpus. Possono coincidere con un morfema, attraversarne i confini o non avere un significato riconoscibile. BPE cerca sequenze frequenti, non analizza la grammatica.

### Esercizio 3 — analisi morfologica e segmentazione

Per `riorganizzazione`, `gattini`, `velocemente` e `casetta`:

1. proponete un'analisi morfologica plausibile, indicando la base e gli affissi quando li riconoscete;
2. indicate quali segmenti potrebbero essere utili come subword a un modello;
3. spiegate perché un merge BPE che produce un segmento simile a un affisso non dimostra che il tokenizer abbia “scoperto” un morfema;
4. segnate i casi in cui la vostra analisi è incerta: non tutte le parole hanno una segmentazione univoca.


## 4. Unicode: caratteri e codifica (introduzione)

Unicode assegna identificatori numerici, detti **code point**, ai caratteri dei sistemi di scrittura e ai simboli. UTF-8 è una codifica che rappresenta i code point come sequenze di uno o più byte. Quindi carattere, code point, byte e token non sono sinonimi. Inoltre, un carattere percepito come unico può essere codificato con più code point, per esempio una lettera e un segno diacritico combinante.

In Python, `len(testo)` conta gli elementi della stringa (code point), non necessariamente i caratteri percepiti dall'utente; `len(testo.encode("utf-8"))` conta i byte. Qui ci fermiamo a questa introduzione: i dettagli successivi della sezione Unicode del capitolo non sono richiesti.

### Esercizio 4 — un segno, più rappresentazioni

Eseguite la cella e confrontate `é` scritto come carattere precomposto e come `e` più accento combinante. Sono uguali come stringhe? Quanti code point e byte UTF-8 hanno? Collegate la risposta alla tokenizzazione: un tokenizer riceve una sequenza codificata, non l'aspetto grafico astratto.


## 5. Corpora: i dati da cui si impara

Un **corpus** è una raccolta di testi scelta per studiare o addestrare un sistema. Può essere composto da documenti, frasi o parole con frequenze associate. Non è una finestra neutrale sulla lingua: lingua, dominio, periodo, genere testuale, provenienza, pulizia e ripetizioni influenzano quali forme e sequenze risultano frequenti.

Nel notebook 02 avete contato occorrenze, tipi e OOV. Per BPE, la distribuzione del corpus decide quali coppie sono candidate ai merge: un tokenizer appreso su testi tecnici può trattare in modo diverso le stesse parole rispetto a uno appreso su conversazioni. Un corpus più grande non è automaticamente migliore se non rappresenta l'uso a cui il tokenizer è destinato.

### Esercizio 5 — scegliere e descrivere un corpus

Immaginate di dover addestrare un tokenizer per un'app di supporto universitario. Confrontate due raccolte: (A) articoli enciclopedici generali; (B) domande anonime di studenti, avvisi e pagine dei corsi.

1. Quali forme e convenzioni potrebbero essere più frequenti in ciascuna raccolta?
2. Quale corpus rappresenta meglio l'app? Quali gruppi o registri potrebbero comunque mancare?
3. Indicate due controlli di qualità da fare prima di usare i testi e spiegate perché duplicati o contenuti fuori dominio altererebbero le frequenze BPE.
4. Proponete un piccolo insieme di parole-test, separato dal corpus usato per imparare i merge, per confrontare la copertura.


## 6. Risorse interattive e documentazione

Usate prima gli strumenti nel browser: non serve installare librerie per le dimostrazioni. Le interfacce possono cambiare nel tempo e non tutti i tokenizer supportano gli stessi modelli.

- [tiktokenizer](https://tiktokenizer.vercel.app/) — visualizza come diversi tokenizer dividono il testo e, quando disponibile, mostra ID e conteggio. 
- [Hugging Face Tokenizer Playground](https://huggingface.co/spaces/Xenova/the-tokenizer-playground) — confronto interattivo con tokenizer associati a modelli Hugging Face.
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer) — dimostrazione ufficiale per osservare la tokenizzazione dei modelli OpenAI.
- [Documentazione Hugging Face: algoritmi di tokenizzazione](https://huggingface.co/docs/transformers/main/en/tokenizer_summary) — panoramica e confronto tra BPE, WordPiece, Unigram e SentencePiece.
- [Repository `tiktoken`](https://github.com/openai/tiktoken) — implementazione open source e modulo didattico per visualizzare i passaggi di BPE.

### Attività — esplorare i tokenizer nel browser

Aprite tiktokenizer e, se disponibile, il playground Hugging Face. Usate gli stessi input in ogni strumento e annotate modello/tokenizer selezionato, segmenti mostrati e numero di token.

1. Confrontate una parola frequente e una forma flessa o derivata: `casa`, `casette`, `tokenizzazione`, `tokenizzazioni`.
2. Confrontate una parola rara o inventata: `superinterfacciabilità` oppure una parola creata da voi. È OOV per intero? Come viene suddivisa?
3. Provate spazi e punteggiatura: `ciao`, ` ciao`, `ciao!`, `ciao !`. Che cosa cambia?
4. Provate Unicode e lingue diverse: `perché`, `caffè`, `🙂`, `東京`. Notate che un carattere visibile può corrispondere a più byte o token.
5. Ripetete con due tokenizer/modelli diversi. Quali risultati restano uguali e quali cambiano?
6. Annotate una somiglianza e una differenza tra tokenizer; riprenderete il confronto con i merge didattici dopo averli programmati.

Gli ID e le segmentazioni variano a seconda del tokenizer scelto e possono cambiare con gli aggiornamenti delle risorse: annotate sempre il nome del tokenizer, non solo il risultato.

## 7. L'idea di BPE

**Byte Pair Encoding (BPE)** costruisce un vocabolario iterativamente. Nella versione semplificata usata qui:

1. ogni parola diventa una sequenza di caratteri;
2. si contano le coppie di caratteri adiacenti, rispettando i confini tra parole;
3. si fonde la coppia più frequente in ogni sua occorrenza non sovrapposta;
4. si ripete per un numero deciso di merge.

Le coppie apprese dipendono dal corpus e dalle regole di pareggio. Non c'è garanzia che un merge corrisponda a un morfema. Le varianti pratiche di BPE possono differire nei dettagli: qui costruiamo un modello trasparente per capire il meccanismo, non per replicare un tokenizer commerciale.

### Dimostrazione — rappresentare parole come sequenze

Per non fondere accidentalmente la fine di una parola con l'inizio della successiva, teniamo una sequenza distinta per ogni parola e conserviamo la frequenza delle parole nel corpus.


In [ ]:
parole = ["casa", "casetta", "casetta", "casale"]
frequenze = {parola: parole.count(parola) for parola in set(parole)}
sequenze = {parola: list(parola) for parola in frequenze}

print("Frequenze:", frequenze)
print("Sequenze:", sequenze)


### Esercizio 6 — contare le coppie

Completate `conta_coppie(sequenze, frequenze)`. Deve sommare le occorrenze di ogni coppia adiacente, moltiplicandole per la frequenza della parola; non deve creare coppie tra parole diverse.

Provate il risultato con le parole della dimostrazione e verificate a mano almeno una coppia. Che effetto ha la parola ripetuta due volte?


### Esercizio 7 — merge non sovrapposti

Scrivete `applica_merge(sequenza, coppia)`: restituisce una nuova lista in cui le occorrenze adiacenti della coppia vengono fuse. Una fusione consuma entrambi gli elementi; quindi le occorrenze sovrapposte non possono essere fuse entrambe nello stesso passaggio.

Provate almeno questi casi: `['a', 'b', 'a', 'b']` con `('a', 'b')`, e `['a', 'a', 'a']` con `('a', 'a')`.


### Esercizio 8 — eseguire un merge a mano e in codice

1. Usate `conta_coppie` per trovare la coppia più frequente nel corpus della dimostrazione.
2. Applicate il merge a tutte le sequenze, conservando le frequenze.
3. Stampate sequenze prima/dopo e ricontate le coppie.
4. Ripetete con una seconda coppia a scelta. Annotate l'ordine dei merge: l'ordine è parte del tokenizer appreso.

In caso di parità, scegliete una regola deterministica e scrivetela: per esempio ordinare le coppie lessicograficamente e prendere la prima.


## 8. Imparare una sequenza di merge

Una singola fusione illustra il meccanismo, ma BPE ne impara molte. L'ordine appreso viene poi riutilizzato per segmentare parole nuove. Il codice seguente generalizza gli esercizi precedenti, mantenendo espliciti i passaggi.

In [ ]:
def impara_bpe(parole, numero_merge):
    frequenze = {}
    for parola in parole:
        frequenze[parola] = frequenze.get(parola, 0) + 1
    sequenze = {parola: list(parola) for parola in frequenze}
    merge_appresi = []

    for _ in range(numero_merge):
        conteggi = conta_coppie(sequenze, frequenze)
        coppia = scegli_coppia(conteggi)
        if coppia is None:
            break
        sequenze = {
            parola: applica_merge(token, coppia)
            for parola, token in sequenze.items()
        }
        merge_appresi.append(coppia)

    return merge_appresi, sequenze, frequenze

corpus = ["casa", "casetta", "casetta", "casale"]
merge_appresi, sequenze_finali, frequenze = impara_bpe(corpus, 5)
print("Merge:", merge_appresi)
print("Segmentazioni apprese:", sequenze_finali)


### Esercizio 9 — seguire e verificare l'apprendimento

1. Eseguite il codice con 1, 2, 4 e 8 merge. Come cambiano le segmentazioni?
2. Aggiungete più occorrenze di una parola e ripetete. Quali merge cambiano?
3. Provate un corpus con parole che condividono prefissi e suffissi, poi uno con parole senza parti ricorrenti.
4. Verificate che la somma del numero di parole pesate nel corpus non cambi durante i merge.
5. Spiegate perché `numero_merge` è una scelta di granularità e non un parametro “linguisticamente corretto” universale.


## 9. Applicare i merge a parole non viste

Per segmentare una parola nuova, iniziamo dai suoi caratteri e applichiamo i merge **nell'ordine in cui sono stati appresi**. Nel nostro esempio didattico, un carattere mai comparso nel corpus resta disponibile come unità singola: questo evita l'OOV completo, ma non è ciò che fanno tutte le implementazioni reali. Le varianti byte-level possono rappresentare qualsiasi sequenza di byte UTF-8, a costo di possibili sequenze più lunghe.

In [ ]:
def segmenta(parola, merge_appresi):
    sequenza = list(parola)
    for coppia in merge_appresi:
        sequenza = applica_merge(sequenza, coppia)
    return sequenza

merge_appresi, _, _ = impara_bpe(corpus, 5)
for parola in ["casa", "casetta", "casette", "casanova", "cane"]:
    print(parola, "->", segmenta(parola, merge_appresi))


### Esercizio 10 — generalizzazione e OOV

Usate i merge del corpus `casa`, `casetta`, `casale` per segmentare `casette`, `casanova` e `cane`.

1. Quali parole sono OOV a livello di parola?
2. Quali unità subword si possono comunque produrre?
3. Provate una parola con un carattere non visto e osservate che la dimostrazione la lascia come unità singola. Perché questo non basta a descrivere tutti i tokenizer moderni?
4. Confrontate la lunghezza della sequenza subword con una unità per parola. Che relazione vedete con lo spazio di contesto di un modello?


## 10. Il corpus influenza il tokenizer

Un tokenizer BPE non scopre una segmentazione universale: le fusioni riflettono frequenze e contenuti del corpus di addestramento. Un corpus tecnico, uno di conversazioni e uno in un'altra lingua possono produrre merge diversi. Perciò la segmentazione può essere efficiente senza rappresentare la struttura morfologica nel modo atteso da una persona.

### Esercizio 11 — confronto tra domini

Create due corpora piccoli: uno di frasi quotidiane e uno di frasi tecniche/informatiche. Inserite ripetizioni realistiche e usate lo stesso numero di merge.

1. Confrontate i merge appresi e segmentate almeno tre parole presenti in entrambi i domini.
2. Cercate una parola frequente in un corpus ma rara o assente nell'altro.
3. Spiegate come la distribuzione delle frequenze cambia i pezzi disponibili.
4. Non interpretate automaticamente i segmenti come morfemi: motivate questa cautela con un esempio.


## 11. Confronto misurabile con il livello parola

Nel notebook 02 avete già confrontato convenzioni e tokenizer tradizionali. Qui riutilizziamo quell'idea per misurare un diverso obiettivo: quante unità servono a rappresentare un insieme di parole, e quali parole restano fuori. Il numero di token non dice da solo quale tokenizer sia “migliore”: occorre considerare il compito, il corpus e il costo degli errori.

### Esercizio 12 — OOV e lunghezza delle rappresentazioni

Usate un corpus di addestramento di almeno 8 parole e una lista di almeno 5 parole di test, includendo forme flesse, composti e almeno una parola inventata o rara.

1. Calcolate la copertura a livello di parola e il numero di parole OOV, riutilizzando le idee del notebook 02.
2. Imparate una sequenza di merge e segmentate le stesse parole di test.
3. Calcolate il numero medio di subword per parola e identificate la parola con la sequenza più lunga.
4. Confrontate i risultati per due valori di `numero_merge`.
5. Scrivete una conclusione: quale tokenizer usereste per una ricerca lessicale? E per un modello generativo? Motivate la risposta senza assumere che meno token sia sempre meglio.


## 12. Laboratorio conclusivo: progettare una rappresentazione

Lavorate in coppia o individualmente su un mini-dominio (per esempio cucina, linguistica, sport, informatica o un insieme di messaggi). Usate solo dati inventati o non sensibili.

### Esercizio 13 — mini-progetto BPE

1. Raccogliete un corpus di almeno 10 forme; includete varianti e ripetizioni. Annotate perché avete scelto quelle frequenze.
2. Imparate 3, 6 e 10 merge e conservate il risultato di ogni prova.
3. Segmentate forme osservate e almeno 5 forme nuove.
4. Confrontate il vocabolario parola intera con le sequenze subword: OOV, lunghezza media e casi sorprendenti.
5. Cambiate una sola scelta per esperimento (frequenze, composizione del corpus oppure numero di merge) e descrivete l'effetto.
6. Consegnate una breve relazione con corpus, risultati, un limite del vostro esperimento e una raccomandazione per il caso d'uso scelto.